# Orden de reducción y error de redondeo

## Pregunta y continuidad
La línea MOMENTS ya permite fusionar particiones mediante Chan después de rebasing dinámico. En reales, la fusión representa una operación asociativa; en binary64, cada suma y producto se redondea y el árbol puede cambiar el resultado. Comparamos el fold secuencial, el orden inverso y un árbol balanceado fijo. Este experimento es un audit de ingeniería; no propone una nueva estimación.

## Fórmula y brecha
Para resúmenes A,B de particiones disjuntas, con medias $\mu_A,\mu_B$, conteos $n_A,n_B$ y sumas centradas $Q_A,Q_B$:
$$\delta=\mu_B-\mu_A,\quad \mu=\mu_A+\delta n_B/n,\quad Q=Q_A+Q_B+\delta^2 n_An_B/n.$$
La derivación expande $\sum (x_i-\mu)^2$ por grupo; los términos cruzados desaparecen por centrado. En máquina, Chan permanece estable frente a la fórmula ingenua, pero no vuelve bitwise asociativo al operador. La herramienta fija una estructura reproducible: pares adyacentes se fusionan por nivel y un resumen impar pasa al nivel siguiente.

## Hipótesis falsable y diseño
En entradas sintéticas alrededor de $10^8$, con desviación de orden $10^{-2}$, el árbol balanceado debe quedar dentro de `max(4 ulp, 1e-12 relativo)` del oráculo Decimal de los mismos float64. El benchmark compara tres órdenes, 3 semillas y n=256/4096, chunks de 32. Tiempo es sólo el merge tree, excluye lectura y construcción de chunks. La semilla y la configuración están fijadas en `config_reduction.json`.

## Límites
El orden balanceado determina un resultado repetible para el mismo orden de chunks, no invariancia a permutaciones ni error universal. Se asume que los resúmenes son disjuntos; la API no deduplica. Los datos son sintéticos, no evidencia empírica.

In [2]:
from src.benchmark_reduction import run
summary = run()
assert summary["rows"] == 18

{"all_within_declared_tolerance": true, "chunk_size": 32, "data": "synthetic Gaussian around 1e8; known scale 0.01; not empirical", "max_tree_spread_sd": 6.938893903907228e-18, "methods": ["left", "reverse", "balanced"], "oracle_error_abs_max": 5.204170427930421e-18, "oracle_error_abs_median": 8.673617379884035e-19, "oracle_error_abs_min": 0.0, "rows": 18, "seeds": [1104, 1105, 1106], "sizes": [256, 4096]}


La figura compara desviaciones estándar finales para tres semillas. Diferencias visibles reflejan sensibilidad al orden bajo precisión finita, no diferencias en el estimando poblacional.

In [4]:
import json
from pathlib import Path
print((Path('results/reduction_summary.json')).read_text())
assert Path('results/reduction_order.png').exists()

{
  "seeds": [
    1104,
    1105,
    1106
  ],
  "sizes": [
    256,
    4096
  ],
  "chunk_size": 32,
  "methods": [
    "left",
    "reverse",
    "balanced"
  ],
  "rows": 18,
  "max_tree_spread_sd": 6.938893903907228e-18,
  "oracle_error_abs_min": 0.0,
  "oracle_error_abs_median": 8.673617379884035e-19,
  "oracle_error_abs_max": 5.204170427930421e-18,
  "all_within_declared_tolerance": true,
  "data": "synthetic Gaussian around 1e8; known scale 0.01; not empirical"
}

